# 04 - PhoBERT Extractive Sanity Notebook

Notebook goals:
- Validate `phobert-extractive` behavior on a small subset before the official benchmark.
- Compare quickly against `tfidf` and `textrank` under the same preprocessing pipeline.
- Inspect qualitative outputs and core metrics (ROUGE/compression/repetition).

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parents[0]
if (ROOT / "backend").exists() is False:
    ROOT = Path.cwd().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))
if str(ROOT / "evaluation") not in sys.path:
    sys.path.insert(0, str(ROOT / "evaluation"))

from app.services.input import process_from_text
from app.services.summarization.summary_service import summarize_processed_input_raw
from evaluation.evaluator import Evaluator
from scripts.shared.io_dataset import load_benchmark_validation_subset, load_split

AttributeError: module 'pyarrow' has no attribute '__version__'

In [ ]:
PROTOCOL_VERSION_EXPECTED = "phase0_v2"
TARGET_SPLIT = "validation"
SEED = 42
SUBSET_LIMIT = 30
ARTICLE_CHAR_THRESHOLD = 1200
TOP_K = 3
ENGINES = ["tfidf", "textrank", "phobert-extractive"]

processed_dir = ROOT / "data" / "processed" / "vietnews"
df, manifest = load_split(processed_dir, TARGET_SPLIT, PROTOCOL_VERSION_EXPECTED)
subset_df, protocol_version, manifest_path, split_path = load_benchmark_validation_subset(
    df,
    manifest=manifest,
    processed_dir=processed_dir,
    target_split=TARGET_SPLIT,
    seed=SEED,
    subset_limit=SUBSET_LIMIT,
    article_char_threshold=ARTICLE_CHAR_THRESHOLD,
)
subset_df[["guid", "article_char_len", "reference_char_len"]].head()

In [ ]:
evaluator = Evaluator(use_stemmer=False)
records = []

for engine in ENGINES:
    for row in subset_df.to_dict(orient="records"):
        article = str(row.get("article", ""))
        reference = str(row.get("reference_summary", ""))
        if not article.strip() or not reference.strip():
            continue

        processed = process_from_text(article)
        t0 = time.perf_counter()
        selected_sentences, engine_meta = summarize_processed_input_raw(
            processed,
            max_sentences=TOP_K,
            ratio=None,
            engine_name=engine,
        )
        latency_sec = time.perf_counter() - t0
        predicted_summary = " ".join(s.strip() for s in selected_sentences if s.strip())
        bundle = evaluator.evaluate_one(
            source_text=processed.cleaned_text,
            reference_summary=reference,
            predicted_summary=predicted_summary,
            latency_sec=latency_sec,
            extra={
                "guid": row.get("guid"),
                "engine": engine,
                "predicted_summary": predicted_summary,
                "selected_indices": engine_meta.get("selected_indices", []),
            },
        )
        records.append(bundle.as_dict())

detail_df = pd.DataFrame(records)
summary_df = (
    detail_df.groupby("engine", as_index=False)
    .agg(
        n=("engine", "count"),
        rouge1_f=("rouge1_f", "mean"),
        rouge2_f=("rouge2_f", "mean"),
        rougeL_f=("rougeL_f", "mean"),
        latency_sec=("latency_sec", "mean"),
        compression_ratio=("compression_ratio", "mean"),
        repetition_rate=("repetition_rate", "mean"),
    )
    .sort_values("rougeL_f", ascending=False)
    .reset_index(drop=True)
)
summary_df

In [ ]:
sample_guid = detail_df[detail_df["engine"] == "phobert-extractive"].head(1)["guid"].tolist()
if sample_guid:
    g = sample_guid[0]
    print("GUID:", g)
    source_row = subset_df[subset_df["guid"].astype(str) == str(g)].head(1).to_dict(orient="records")[0]
    print("\nArticle snippet:")
    print(str(source_row.get("article", ""))[:800])
    print("\nReference:")
    print(str(source_row.get("reference_summary", ""))[:500])
    print("\nPredictions:")
    for engine in ENGINES:
        pred = detail_df[(detail_df["guid"].astype(str) == str(g)) & (detail_df["engine"] == engine)].head(1)
        if not pred.empty:
            print(f"- {engine}:", str(pred.iloc[0]["predicted_summary"])[:500])